# Trabalho 1 — Agente único de análise rodoviária

**Engenharia de Agentes e IA Agêntica** · Pós-graduação · PUC Minas
**André Oliveira** · Entrega 1 · agente único, integração direta

O notebook caracteriza o caso e implementa a primeira versão funcional da solução: um agente baseado em LLM, com ciclo de decisão, ação e observação, acesso a conhecimento e ferramentas chamadas pelo runtime local. Não usa MCP nem LangGraph. O segredo externo é a chave `GEMINI_API_KEY`. Os CSVs e o conhecimento não entram no Git: a execução baixa as duas pastas públicas do Drive.

| | |
|---|---|
| Ambiente | Google Colab, com a pasta desta entrega como diretório de trabalho |
| Modelo | o primeiro Flash que responder na lista da seção 5 |
| Conhecimento | trechos `.txt` na pasta pública do Google Drive |
| Fatos | `ticket_viagem.csv` e `ocupacao.csv` na pasta pública do Google Drive |
| Saída | `parecer_analista.csv` |


---

# 1. Caracterização do caso

## 1.1 O problema

Um analista de malha rodoviária recebe perguntas que misturam venda e operação sobre a mesma viagem. Quer saber se a linha Belo Horizonte–Rio de Janeiro, num dia concreto, tem problema de receita, de lotação, ou se a pergunta nem é um fato e sim uma regra de leitura.

Os dados existem em dois grãos que não se somam:

- o bilhete, em `ticket_viagem.csv`, responde receita, quantidade de passagens, canal, antecedência e passageiro-quilômetro;
- a perna física, em `ocupacao.csv`, responde ocupantes, capacidade, vagas, embarque, desembarque e excesso.

A pergunta de demonstração usa estes termos:

- **Bilhete:** uma passagem vendida. Receita e passageiro-quilômetro ficam neste grão.
- **Perna:** o trecho físico entre duas paradas consecutivas. A lotação é ocupantes divididos pela capacidade daquela perna.
- **ASK:** assentos ofertados multiplicados pelo quilômetro oficial do serviço.
- **IAP:** distância transportada nos bilhetes dividida pelo ASK, uma razão entre 0 e 1. Sem o quilômetro oficial, o IAP desta entrega fica como pendência.

## 1.2 Por que um agente, e não um fluxo fixo

Um script determinístico precisaria escolher o arquivo, o filtro e a agregação antes de ver a pergunta. Aqui a pergunta é que decide:

- definição de IAP ou de grão pede a base de conhecimento, e não um número;
- receita pede a base de vendas;
- "onde o ônibus encheu" pede a base de ocupação;
- uma linha ou data que não está na base pede a consulta e, em seguida, a declaração de ausência;
- uma pergunta de negócio pode precisar da regra e do fato, nessa ordem, e parar quando a evidência basta.

Automatizar as três consultas sempre, para toda pergunta, mistura grãos, gasta contexto e ainda não diz qual número responde à pergunta. O agente percebe a pergunta, escolhe a próxima ferramenta, observa o retorno e decide de novo. Quem recebe o parecer continua responsável pela decisão de negócio. O agente recomenda.

## 1.3 De onde vêm os fatos

Os dois CSVs completos somam cerca de 730 MB e não cabem no Git gratuito. A execução baixa esta pasta pública:

https://drive.google.com/drive/folders/1-1ckwMQA2OUvt0gMWf3pp5yx1PMStn2I

- `ticket_viagem.csv`, um bilhete por linha;
- `ocupacao.csv`, uma perna por linha;
- empresa `modal`, várias linhas e várias datas.

A demonstração pergunta pela linha `mg/belo horizonte - rj/rio de janeiro` em `2026-04-05`. Linha ou data inexistente devolve ausência, e o agente não troca o filtro para achar um número.

## 1.4 O que esta entrega não é

Não é sistema multiagente, não usa framework de orquestração e não implementa governança da unidade 6. O servidor MCP da aula 3 fica de fora: as ferramentas são funções Python registradas num dicionário, e o runtime valida, executa e devolve a observação.


---

# 2. Ambiente, PEAS e ciclo

```
Pergunta do analista
        │
        ▼
   perceber()
        │
        ▼
   decidir()  ── pede uma ferramenta, ou encerra
        │
        ▼
   runtime valida o contrato e executa
        │
        ├── consultar_base_de_conhecimento   regras de grão e IAP
        ├── consultar_vendas                 fato do bilhete
        └── consultar_ocupacao               fato da perna
        │
        ▼
   observação volta ao contexto
        │
        └── repete até evidência suficiente, ausência ou limite
        │
        ▼
   parecer estruturado
        │
        ▼
   parecer_analista.csv
```

| PEAS | Neste agente |
|---|---|
| Performance | Grão certo, número só com evidência da ferramenta, ausência explícita |
| Environment | Recorte de vendas e ocupação, mais os trechos de conhecimento |
| Actuators | Inclusão de uma linha em `parecer_analista.csv` |
| Sensors | Texto da pergunta e o retorno estruturado de cada ferramenta |



---

# 3. Preparação

No Colab basta o notebook. Os CSVs e os trechos de `conhecimento/` vêm de duas pastas do Drive. As duas precisam estar compartilhadas como qualquer pessoa com o link. A chave fica no secret `GEMINI_API_KEY` e não é escrita no arquivo. A primeira execução baixa cerca de 730 MB dos CSVs.


In [ ]:
%pip install -q "google-genai==2.29.0" pandas numpy jsonschema gdown


In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from jsonschema import Draft202012Validator
from jsonschema.exceptions import ValidationError


URL_PASTA_DRIVE = "https://drive.google.com/drive/folders/1-1ckwMQA2OUvt0gMWf3pp5yx1PMStn2I"
URL_PASTA_CONHECIMENTO = "https://drive.google.com/drive/folders/1TKIUYyCiG6UoHt8ihUyKsdmzyZwYdfHY"
ARQUIVOS_CSV = ("ticket_viagem.csv", "ocupacao.csv")


def localizar_pasta() -> Path:
    """Pasta local só para gravar o parecer. Os insumos vêm do Drive."""
    bases = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    nomes = (
        Path("."),
        Path("Trabalho1_Eng_IA_AndreOliveira"),
        Path("Eng_Agentes_IA") / "Trabalho1_Eng_IA_AndreOliveira",
    )
    for base in bases:
        for nome in nomes:
            candidata = (base / nome).resolve()
            if (candidata / "Trabalho1_Agente_Unico_Rodoviario.ipynb").exists():
                return candidata
    return Path.cwd().resolve()


PASTA = localizar_pasta()
print("Pasta da entrega:", PASTA)


---

# 4. O ambiente que o agente enxerga

A leitura segue a aula 1: tudo entra como texto e a conversão numérica é explícita. A célula baixa os CSVs do Drive, se ainda não estiverem em cache, e mede a base. Esses números são o panorama do ambiente, não ainda uma decisão do agente.


In [ ]:
def baixar_csvs() -> Path:
    """Baixa os dois CSVs da pasta pública do Drive para um cache fora do Git."""
    destino = Path("/tmp/trabalho1_rodoviario_csvs")
    destino.mkdir(parents=True, exist_ok=True)
    prontos = [
        (destino / nome).exists() and (destino / nome).stat().st_size > 1_000_000
        for nome in ARQUIVOS_CSV
    ]
    if all(prontos):
        print("CSVs já estão em", destino)
        return destino

    import gdown

    print("Baixando os CSVs do Google Drive. São cerca de 730 MB.")
    print(URL_PASTA_DRIVE)
    gdown.download_folder(url=URL_PASTA_DRIVE, output=str(destino), quiet=False)
    faltando = [nome for nome in ARQUIVOS_CSV if not (destino / nome).exists()]
    if faltando:
        raise FileNotFoundError(
            f"A pasta do Drive não devolveu {faltando}. Confira o compartilhamento: {URL_PASTA_DRIVE}"
        )
    return destino


def carregar_tabela(pasta_csv: Path, nome: str) -> pd.DataFrame:
    """Lê um CSV sem deixar o pandas inventar tipo nem NaN."""
    return pd.read_csv(pasta_csv / nome, dtype=str, keep_default_na=False)


def _numeros(tabela: pd.DataFrame, colunas: list[str]) -> None:
    for coluna in colunas:
        tabela[coluna] = pd.to_numeric(tabela[coluna], errors="coerce").fillna(0)


PASTA_CSV = baixar_csvs()
VENDAS = carregar_tabela(PASTA_CSV, "ticket_viagem.csv")
OCUPACAO = carregar_tabela(PASTA_CSV, "ocupacao.csv")
_numeros(VENDAS, ["receita", "pax_km", "dias_antecedencia"])
_numeros(
    OCUPACAO,
    [
        "capacidade",
        "ocupantes",
        "embarques_origem",
        "desembarques_destino",
        "excesso",
        "vagas",
        "sem_bilhete",
    ],
)

LINHA_AMOSTRA = "mg/belo horizonte - rj/rio de janeiro"
DATA_AMOSTRA = "2026-04-05"

print(f"Origem dos CSVs: {PASTA_CSV}")
print(f"Bilhetes na base: {len(VENDAS):,}")
print(f"Pernas na base:   {len(OCUPACAO):,}")
print(f"Linhas distintas: {VENDAS['linha'].nunique()}")

caso_vendas = VENDAS[
    (VENDAS["linha"] == LINHA_AMOSTRA) & VENDAS["data_viagem"].str.startswith(DATA_AMOSTRA)
]
caso_ocupacao = OCUPACAO[
    (OCUPACAO["linha"] == LINHA_AMOSTRA) & OCUPACAO["data_viagem"].str.startswith(DATA_AMOSTRA)
]
print(f"Caso da demonstração: {LINHA_AMOSTRA} em {DATA_AMOSTRA}")
print(f"Bilhetes do caso: {len(caso_vendas):,}  |  receita R$ {caso_vendas['receita'].sum():,.2f}")
capacidade = caso_ocupacao["capacidade"].sum()
ocupantes = caso_ocupacao["ocupantes"].sum()
taxa = 100 * ocupantes / capacidade if capacidade else 0
print(f"Pernas do caso: {len(caso_ocupacao):,}  |  lotação {taxa:.1f}%")
print("Essa lotação não é o IAP. A regra está na base de conhecimento.")


---

# 5. Conhecimento

Os trechos vêm desta pasta do Drive: https://drive.google.com/drive/folders/1TKIUYyCiG6UoHt8ihUyKsdmzyZwYdfHY

Cada arquivo `.txt` é um chunk, com código estável para citação: serviço, bilhete, perna, SKU, IAP e o roteamento entre ferramentas.

A célula abaixo fixa a ordem dos modelos Flash. Na seção 9 o notebook testa um por um e fica com o primeiro que responder, porque a disponibilidade do Flash oscila.

In [ ]:
DOCUMENTOS: list[dict[str, str]] = []
INDICE: list[dict[str, Any]] = []
CLIENTE: genai.Client | None = None

MODELOS_FLASH = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-flash-latest",
]
MODELO = MODELOS_FLASH[0]
MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768
TEMPERATURA = 0.1
ORCAMENTO_DE_RACIOCINIO = 0
USAR_ORCAMENTO_DE_RACIOCINIO = True


def config_de_geracao(**extras: Any) -> types.GenerateContentConfig:
    """Monta a chamada. Alguns Flash recusam thinking_budget e seguem sem ele."""
    if USAR_ORCAMENTO_DE_RACIOCINIO:
        extras["thinking_config"] = types.ThinkingConfig(thinking_budget=ORCAMENTO_DE_RACIOCINIO)
    return types.GenerateContentConfig(**extras)
ESPERAS_ENTRE_TENTATIVAS = (30, 60)
MAX_ITERACOES = 5
TOP_K = 3


def baixar_conhecimento() -> Path:
    """Baixa os .txt da pasta de conhecimento do Drive para um cache fora do Git."""
    destino = Path("/tmp/trabalho1_rodoviario_conhecimento")
    destino.mkdir(parents=True, exist_ok=True)
    existentes = [caminho for caminho in destino.rglob("*.txt") if caminho.stat().st_size > 0]
    if existentes:
        print("Conhecimento já está em", destino)
        return destino

    import gdown

    print("Baixando a base de conhecimento do Google Drive.")
    print(URL_PASTA_CONHECIMENTO)
    try:
        gdown.download_folder(url=URL_PASTA_CONHECIMENTO, output=str(destino), quiet=False)
    except Exception as erro:
        raise RuntimeError(
            "Não consegui baixar o conhecimento. Compartilhe a pasta como "
            "'Qualquer pessoa com o link'. " + URL_PASTA_CONHECIMENTO
        ) from erro
    if not list(destino.rglob("*.txt")):
        raise FileNotFoundError(
            "A pasta do Drive não devolveu arquivos .txt. " + URL_PASTA_CONHECIMENTO
        )
    return destino


def carregar_conhecimento() -> list[dict[str, str]]:
    """Lê os trechos de conhecimento baixados do Drive."""
    documentos = []
    for caminho in sorted(baixar_conhecimento().rglob("*.txt")):
        texto = caminho.read_text(encoding="utf-8").strip()
        titulo = caminho.stem
        for linha in texto.splitlines():
            if linha.startswith("Título:"):
                titulo = linha.split(":", 1)[1].strip()
                break
        documentos.append({
            "codigo": caminho.stem.split("_", 1)[0],
            "titulo": titulo,
            "conteudo": texto,
        })
    return documentos


def _tokens(texto: str) -> set[str]:
    bruto = "".join(caracter.lower() if caracter.isalnum() else " " for caracter in texto)
    vazios = {"de", "da", "do", "das", "dos", "a", "o", "e", "que", "em", "para", "com", "nao", "não", "um", "uma"}
    return {token for token in bruto.split() if len(token) > 2 and token not in vazios}


def buscar_lexical(pergunta: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    """Recupera trechos pelo overlap de termos. É o modo de reserva."""
    termos = _tokens(pergunta)
    pontuados = []
    for documento in DOCUMENTOS:
        comuns = termos & _tokens(documento["titulo"] + " " + documento["conteudo"])
        if comuns:
            pontuados.append((len(comuns), documento))
    pontuados.sort(key=lambda item: item[0], reverse=True)
    return [
        {
            "codigo": documento["codigo"],
            "titulo": documento["titulo"],
            "trecho": documento["conteudo"][:700],
            "score": float(pontos),
        }
        for pontos, documento in pontuados[:top_k]
    ]


def _normalizar(valores: list[float]) -> np.ndarray:
    vetor = np.asarray(valores, dtype=np.float32)
    norma = float(np.linalg.norm(vetor))
    if norma == 0:
        return vetor
    return vetor / norma


def _embed(textos: list[str], tipo_de_tarefa: str) -> list[np.ndarray]:
    """Uma tentativa. A reserva lexical cobre a falha, sem esperar o retry longo."""
    if CLIENTE is None:
        raise RuntimeError("Cliente de embedding ainda não foi criado.")
    resposta = CLIENTE.models.embed_content(
        model=MODELO_EMBEDDING,
        contents=textos,
        config=types.EmbedContentConfig(
            task_type=tipo_de_tarefa,
            output_dimensionality=DIMENSAO_EMBEDDING,
        ),
    )
    return [_normalizar(item.values) for item in resposta.embeddings]


def indexar_conhecimento() -> str:
    """Constrói o índice vetorial. Devolve o modo que ficou ativo."""
    global INDICE
    INDICE = []
    try:
        vetores = _embed([doc["conteudo"] for doc in DOCUMENTOS], "RETRIEVAL_DOCUMENT")
    except Exception as erro:
        print(f"Embedding indisponível ({type(erro).__name__}). Conhecimento fica em modo lexical.")
        return "lexical"
    INDICE = [
        {**documento, "vetor": vetor}
        for documento, vetor in zip(DOCUMENTOS, vetores)
    ]
    print(f"Índice vetorial pronto: {len(INDICE)} trechos.")
    return "vetorial"


def buscar_vetorial(pergunta: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    """Compara a pergunta aos trechos por similaridade de cosseno."""
    consulta = _embed([pergunta], "RETRIEVAL_QUERY")[0]
    pontuados = []
    for documento in INDICE:
        score = float(np.dot(consulta, documento["vetor"]))
        pontuados.append((score, documento))
    pontuados.sort(key=lambda item: item[0], reverse=True)
    return [
        {
            "codigo": documento["codigo"],
            "titulo": documento["titulo"],
            "trecho": documento["conteudo"][:700],
            "score": round(score, 4),
        }
        for score, documento in pontuados[:top_k]
        if score >= 0.25
    ]


def buscar_conhecimento(pergunta: str, top_k: int = TOP_K) -> tuple[list[dict[str, Any]], str]:
    """Tenta a busca vetorial e cai para a lexical se ela não produzir evidência."""
    if INDICE:
        try:
            achados = buscar_vetorial(pergunta, top_k)
            if achados:
                return achados, "vetorial"
        except Exception as erro:
            print(f"Busca vetorial falhou ({type(erro).__name__}). Usando modo lexical.")
    return buscar_lexical(pergunta, top_k), "lexical"


DOCUMENTOS = carregar_conhecimento()
print("Trechos de conhecimento:", ", ".join(doc["codigo"] for doc in DOCUMENTOS))


---

# 6. Ferramentas, em integração direta

Há três capacidades. O modelo só vê nome, descrição e schema. O runtime valida o argumento contra o schema, executa a função e devolve um retorno curto. Exceção de execução vira observação com `status: error`, para o agente poder tentar de novo em vez de derrubar o ciclo.

| Ferramenta | Quando usar | O que não responde |
|---|---|---|
| `consultar_base_de_conhecimento` | Regra de grão, definição de IAP, limite da amostra | Receita ou lotação de um dia |
| `consultar_vendas` | Bilhete, receita, canal, antecedência, passageiro-quilômetro | Lotação da perna |
| `consultar_ocupacao` | Ocupantes, vagas, embarque, excesso, taxa de lotação da perna | Faturamento e IAP |


In [ ]:
def _cidades(texto: str) -> list[str]:
    cidades = []
    for lado in str(texto).lower().split(" - "):
        lado = lado.strip()
        if "/" in lado:
            lado = lado.split("/", 1)[1].strip()
        if lado:
            cidades.append(lado)
    return cidades


def _resolver_linha(texto: str) -> str | None:
    """Aceita o nome canônico ou a forma 'belo horizonte - rio de janeiro'."""
    if not str(texto).strip():
        return None
    alvo = " ".join(str(texto).lower().split())
    linhas = sorted(set(VENDAS["linha"]).union(set(OCUPACAO["linha"])))
    for linha in linhas:
        if linha.lower() == alvo:
            return linha
    cidades = _cidades(alvo)
    if len(cidades) < 2:
        return None
    for linha in linhas:
        presentes = _cidades(linha)
        if len(presentes) >= 2 and cidades[0] == presentes[0] and cidades[1] == presentes[1]:
            return linha
    return None


def _data_valida(data: str) -> bool:
    if len(data) != 10 or data[4] != "-" or data[7] != "-":
        return False
    ano, mes, dia = data.split("-")
    return ano.isdigit() and mes.isdigit() and dia.isdigit()


def consultar_base_de_conhecimento(pergunta: str) -> dict[str, Any]:
    """Consulta as regras de grão, IAP e roteamento.

    Use para saber qual fonte responde à pergunta e o que o número significa.
    Não use para obter receita, quantidade de bilhetes ou lotação.
    """
    pergunta = str(pergunta).strip()
    if not pergunta:
        return {
            "status": "invalid_arguments",
            "evidencias": [],
            "observacao": "A pergunta de conhecimento não pode ser vazia.",
        }
    evidencias, modo = buscar_conhecimento(pergunta)
    if not evidencias:
        return {
            "status": "insufficient_evidence",
            "modo_recuperacao": modo,
            "query": pergunta,
            "evidencias": [],
            "observacao": "Nenhum trecho de conhecimento cobriu a pergunta.",
        }
    return {
        "status": "success",
        "modo_recuperacao": modo,
        "query": pergunta,
        "evidencias": evidencias,
    }


def consultar_vendas(
    linha: str,
    data_viagem: str,
    classe: str = "",
    origem_bilhete: str = "",
    destino_bilhete: str = "",
) -> dict[str, Any]:
    """Consulta vendas no grão do bilhete.

    Use para receita, quantidade de passagens, ticket médio, canal,
    antecedência e passageiro-quilômetro. Data no formato AAAA-MM-DD.
    Não use para lotação, vagas ou definição de IAP.
    """
    if not _data_valida(str(data_viagem)):
        return {
            "status": "invalid_arguments",
            "observacao": "data_viagem deve estar no formato AAAA-MM-DD.",
        }
    linha_real = _resolver_linha(linha)
    if linha_real is None:
        return {
            "status": "not_found",
            "encontrado": False,
            "observacao": "A linha não foi encontrada na base. Não substitua por outra linha.",
        }
    recorte = VENDAS[
        (VENDAS["linha"] == linha_real)
        & VENDAS["data_viagem"].str.startswith(str(data_viagem))
    ].copy()
    if classe.strip():
        recorte = recorte[recorte["classe"].str.lower() == classe.strip().lower()]
    if origem_bilhete.strip():
        recorte = recorte[
            recorte["origem_bilhete"].str.contains(origem_bilhete.strip().lower(), regex=False)
        ]
    if destino_bilhete.strip():
        recorte = recorte[
            recorte["destino_bilhete"].str.contains(destino_bilhete.strip().lower(), regex=False)
        ]
    if recorte.empty:
        return {
            "status": "not_found",
            "encontrado": False,
            "linha": linha_real,
            "data_viagem": data_viagem,
            "observacao": "Não há bilhete nesse recorte. Não substitua por outra data ou outra linha.",
        }
    receita = float(recorte["receita"].sum())
    quantidade = int(len(recorte))
    canais = [
        {"canal": canal, "bilhetes": int(qtd)}
        for canal, qtd in recorte["canal"].value_counts().head(3).items()
    ]
    return {
        "status": "success",
        "encontrado": True,
        "grao": "bilhete",
        "linha": linha_real,
        "data_viagem": data_viagem,
        "quantidade_bilhetes": quantidade,
        "receita_total": round(receita, 2),
        "ticket_medio": round(receita / quantidade, 2),
        "pax_km_total": round(float(recorte["pax_km"].sum()), 1),
        "antecedencia_media_dias": round(float(recorte["dias_antecedencia"].mean()), 1),
        "canais": canais,
        "observacao": "Receita no grão do bilhete. Não rateie este valor entre pernas.",
    }


def consultar_ocupacao(
    linha: str,
    data_viagem: str,
    classe: str = "",
    origem: str = "",
    destino: str = "",
) -> dict[str, Any]:
    """Consulta lotação no grão da perna física.

    Use para ocupantes, capacidade, vagas, embarque, desembarque, excesso
    e taxa de lotação da perna. Não use para receita nem para calcular IAP.
    """
    if not _data_valida(str(data_viagem)):
        return {
            "status": "invalid_arguments",
            "observacao": "data_viagem deve estar no formato AAAA-MM-DD.",
        }
    linha_real = _resolver_linha(linha)
    if linha_real is None:
        return {
            "status": "not_found",
            "encontrado": False,
            "observacao": "A linha não foi encontrada na base. Não substitua por outra linha.",
        }
    recorte = OCUPACAO[
        (OCUPACAO["linha"] == linha_real)
        & OCUPACAO["data_viagem"].str.startswith(str(data_viagem))
    ].copy()
    if classe.strip():
        recorte = recorte[recorte["classe"].str.lower() == classe.strip().lower()]
    if origem.strip():
        recorte = recorte[recorte["origem"].str.contains(origem.strip().lower(), regex=False)]
    if destino.strip():
        recorte = recorte[recorte["destino"].str.contains(destino.strip().lower(), regex=False)]
    if recorte.empty:
        return {
            "status": "not_found",
            "encontrado": False,
            "linha": linha_real,
            "data_viagem": data_viagem,
            "observacao": "Não há perna nesse recorte.",
        }
    capacidade = float(recorte["capacidade"].sum())
    ocupantes = float(recorte["ocupantes"].sum())
    agrupado = (
        recorte.groupby(["origem", "destino"], as_index=False)[["capacidade", "ocupantes"]]
        .sum()
    )
    agrupado["taxa"] = np.where(
        agrupado["capacidade"] > 0,
        100 * agrupado["ocupantes"] / agrupado["capacidade"],
        0,
    )
    mais_cheios = agrupado.sort_values("taxa", ascending=False).head(5)
    return {
        "status": "success",
        "encontrado": True,
        "grao": "perna_operacional",
        "linha": linha_real,
        "data_viagem": data_viagem,
        "quantidade_pernas": int(len(recorte)),
        "capacidade_total": int(capacidade),
        "ocupantes_total": int(ocupantes),
        "vagas_total": int(recorte["vagas"].sum()),
        "embarques_total": int(recorte["embarques_origem"].sum()),
        "desembarques_total": int(recorte["desembarques_destino"].sum()),
        "excesso_total": int(recorte["excesso"].sum()),
        "sem_bilhete_total": int(recorte["sem_bilhete"].sum()),
        "taxa_lotacao_perna_percentual": round(100 * ocupantes / capacidade, 1) if capacidade else None,
        "pernas_mais_cheias": [
            {
                "origem": linha_perna.origem,
                "destino": linha_perna.destino,
                "ocupantes": int(linha_perna.ocupantes),
                "capacidade": int(linha_perna.capacidade),
                "taxa_lotacao_percentual": round(float(linha_perna.taxa), 1),
            }
            for linha_perna in mais_cheios.itertuples(index=False)
        ],
        "iap": "nao_calculavel",
        "observacao": (
            "Taxa de lotação da perna. Não é IAP: o IAP divide a distância "
            "dos bilhetes pelo ASK, e o km oficial não está nesta base."
        ),
    }


SCHEMAS: dict[str, dict[str, Any]] = {
    "consultar_base_de_conhecimento": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "pergunta": {"type": "string", "description": "Pergunta sobre regra, grão ou IAP."},
        },
        "required": ["pergunta"],
    },
    "consultar_vendas": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "linha": {"type": "string"},
            "data_viagem": {"type": "string", "description": "AAAA-MM-DD"},
            "classe": {"type": "string"},
            "origem_bilhete": {"type": "string"},
            "destino_bilhete": {"type": "string"},
        },
        "required": ["linha", "data_viagem"],
    },
    "consultar_ocupacao": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "linha": {"type": "string"},
            "data_viagem": {"type": "string", "description": "AAAA-MM-DD"},
            "classe": {"type": "string"},
            "origem": {"type": "string"},
            "destino": {"type": "string"},
        },
        "required": ["linha", "data_viagem"],
    },
}

FUNCOES: dict[str, Callable[..., dict[str, Any]]] = {
    "consultar_base_de_conhecimento": consultar_base_de_conhecimento,
    "consultar_vendas": consultar_vendas,
    "consultar_ocupacao": consultar_ocupacao,
}

DESCRICOES = {
    "consultar_base_de_conhecimento": consultar_base_de_conhecimento.__doc__ or "",
    "consultar_vendas": consultar_vendas.__doc__ or "",
    "consultar_ocupacao": consultar_ocupacao.__doc__ or "",
}


def executar_ferramenta(nome: str, argumentos: dict[str, Any]) -> dict[str, Any]:
    """Valida o contrato e executa. Falha volta como observação, não como exceção."""
    if nome not in FUNCOES:
        return {
            "status": "error",
            "observacao": f"Ferramenta desconhecida: {nome}.",
        }
    validador = Draft202012Validator(SCHEMAS[nome])
    erros = sorted(validador.iter_errors(argumentos), key=lambda erro: list(erro.path))
    if erros:
        return {
            "status": "invalid_arguments",
            "observacao": "; ".join(erro.message for erro in erros[:3]),
        }
    try:
        return FUNCOES[nome](**argumentos)
    except Exception as erro:
        return {
            "status": "error",
            "observacao": f"{type(erro).__name__}: {erro}",
        }


def ferramenta_gemini() -> types.Tool:
    """Publica os contratos locais como FunctionDeclarations. Não há discovery MCP."""
    declaracoes = [
        types.FunctionDeclaration(
            name=nome,
            description=" ".join(DESCRICOES[nome].split()),
            parameters_json_schema=SCHEMAS[nome],
        )
        for nome in FUNCOES
    ]
    return types.Tool(function_declarations=declaracoes)


print("Ferramentas em integração direta:", ", ".join(FUNCOES))


---

# 7. O agente

`executar()` percorre perceber, o laço de ferramentas e a ação. O laço termina de três jeitos: o modelo pede mais nada e segue para a resposta estruturada; o modelo stub de teste encerra; ou o contador chega a `MAX_ITERACOES` e o parecer sai com `limite_de_execucao`, sem chamar o modelo de novo.

A validação da resposta final confere o contrato e recusa conclusão `respondida` sem uma ferramenta que tenha devolvido `success`. Fonte citada e não chamada também fica registrada em `problemas_de_validacao`.


In [ ]:
PAPEL_DO_AGENTE = """
Você é um agente de análise da malha rodoviária. Prepara um parecer para um analista humano.
Você não decide política comercial nem altera a grade. Você escolhe a evidência e apresenta o número certo.

BASE OPERACIONAL
Os CSVs cobrem várias linhas e várias datas da empresa modal.
Passe a linha e a data que a pergunta citar. Data no formato AAAA-MM-DD.
Quando a pergunta for a da demonstração, use a linha
mg/belo horizonte - rj/rio de janeiro e a data 2026-04-05.
Não troque esse nome por uma variante com sufixo, como rio de janeiro(rio2).
Se a pergunta citar outra linha ou outra data, consulte a ferramenta com esses valores.

COMO DECIDIR
- Regra, definição, grão ou IAP: consultar_base_de_conhecimento.
- Receita, bilhetes, ticket médio, canal, antecedência, passageiro-quilômetro: consultar_vendas.
- Lotação, vagas, embarque, excesso, onde o ônibus encheu: consultar_ocupacao.
- Uma pergunta de negócio pode exigir a regra e depois o fato. Pare quando a evidência bastar.
- Não invente número, linha, data nem percentual de IAP.
- Se o retorno for not_found, insufficient_evidence ou invalid_arguments, diga isso.
  Não troque a linha nem a data para achar um número.
- Não repita a mesma chamada com os mesmos argumentos.
""".strip()


DECISAO_SCHEMA: dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "resposta": {"type": "string"},
        "grao": {
            "type": "string",
            "enum": [
                "bilhete",
                "perna_operacional",
                "regra_de_negocio",
                "misto",
                "indeterminado",
            ],
        },
        "fontes": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": [
                    "consultar_base_de_conhecimento",
                    "consultar_vendas",
                    "consultar_ocupacao",
                ],
            },
        },
        "conclusao": {
            "type": "string",
            "enum": [
                "respondida",
                "evidencia_insuficiente",
                "fora_da_amostra",
                "limite_de_execucao",
            ],
        },
        "pendencias": {"type": "array", "items": {"type": "string"}},
        "justificativa": {"type": "string"},
    },
    "required": ["resposta", "grao", "fontes", "conclusao", "pendencias", "justificativa"],
}


def _gerar_com_retry(conteudos: list[Any], config: types.GenerateContentConfig) -> Any:
    """Repete a chamada ao modelo diante de falha transitória."""
    if CLIENTE is None:
        raise RuntimeError("Cliente do modelo não foi criado.")
    ultimo: Exception | None = None
    for tentativa in range(len(ESPERAS_ENTRE_TENTATIVAS) + 1):
        try:
            return CLIENTE.models.generate_content(
                model=MODELO,
                contents=conteudos,
                config=config,
            )
        except Exception as erro:
            ultimo = erro
            if tentativa < len(ESPERAS_ENTRE_TENTATIVAS):
                espera = ESPERAS_ENTRE_TENTATIVAS[tentativa]
                print(f"Falha na geração ({type(erro).__name__}). Nova tentativa em {espera}s.")
                time.sleep(espera)
    raise RuntimeError(f"Falha ao gerar após as tentativas: {ultimo}")


def validar_decisao(decisao: dict[str, Any], chamadas: list[dict[str, Any]]) -> list[str]:
    """Confere contrato, fonte citada e conclusão sem evidência."""
    problemas = []
    obrigatorios = ("resposta", "grao", "fontes", "conclusao", "pendencias", "justificativa")
    for campo in obrigatorios:
        if campo not in decisao:
            problemas.append(f"Campo ausente: {campo}")
    if problemas:
        return problemas
    erros = sorted(
        Draft202012Validator(DECISAO_SCHEMA).iter_errors(decisao),
        key=lambda erro: list(erro.path),
    )
    problemas.extend(erro.message for erro in erros[:4])
    usadas = {chamada["nome"] for chamada in chamadas}
    for fonte in decisao.get("fontes") or []:
        if fonte not in usadas:
            problemas.append(f"Fonte citada sem chamada: {fonte}")
    sucesso = [chamada for chamada in chamadas if chamada["status"] == "success"]
    if decisao.get("conclusao") == "respondida" and not sucesso:
        problemas.append("Conclusão respondida sem ferramenta com status success.")
    return problemas


class AgenteAnalise:
    """Agente único. O parâmetro decidir existe para os testes do runtime."""

    def __init__(self, max_iteracoes: int = MAX_ITERACOES, verbose: bool = True) -> None:
        self.max_iteracoes = max_iteracoes
        self.verbose = verbose
        self.status = "IDLE"

    def _log(self, texto: str = "") -> None:
        if self.verbose:
            print(texto)

    def perceber(self, pergunta: str) -> dict[str, str]:
        """A percepção desta entrega é a pergunta e a fronteira da amostra."""
        self.status = "PERCEIVING"
        return {
            "pergunta": " ".join(str(pergunta).split()),
            "linha_disponivel": LINHA_AMOSTRA,
            "data_disponivel": DATA_AMOSTRA,
        }

    def _parecer(
        self,
        percepcao: dict[str, str],
        decisao: dict[str, Any],
        chamadas: list[dict[str, Any]],
        problemas: list[str],
        gravar: bool,
    ) -> dict[str, Any]:
        registro = {
            "pergunta": percepcao["pergunta"],
            "resposta": decisao.get("resposta", ""),
            "grao": decisao.get("grao", "indeterminado"),
            "conclusao": decisao.get("conclusao", "evidencia_insuficiente"),
            "fontes": list(decisao.get("fontes") or []),
            "pendencias": list(decisao.get("pendencias") or []),
            "justificativa": decisao.get("justificativa", ""),
            "problemas_de_validacao": problemas,
            "chamadas": chamadas,
        }
        if gravar:
            self._gravar(registro)
        self.status = "IDLE"
        return registro

    def _gravar(self, registro: dict[str, Any]) -> None:
        """A ação que modifica o ambiente: uma linha no parecer do analista."""
        caminho = PASTA / "parecer_analista.csv"
        linha = {
            "pergunta": registro["pergunta"],
            "conclusao": registro["conclusao"],
            "grao": registro["grao"],
            "fontes": " | ".join(registro["fontes"]),
            "resposta": registro["resposta"],
            "pendencias": " | ".join(registro["pendencias"]),
            "problemas_de_validacao": " | ".join(registro["problemas_de_validacao"]),
        }
        novo = not caminho.exists()
        with caminho.open("a", encoding="utf-8") as arquivo:
            if novo:
                arquivo.write(";".join(linha) + "\n")
            arquivo.write(
                ";".join(str(linha[coluna]).replace(";", ",").replace("\n", " ") for coluna in linha)
                + "\n"
            )
        self._log(f"Parecer acrescentado em {caminho.name}")

    def _executar_chamadas(
        self,
        chamadas: list[Any],
        acumuladas: list[dict[str, Any]],
    ) -> list[Any]:
        partes = []
        for chamada in chamadas:
            if isinstance(chamada, dict):
                nome = chamada["nome"]
                argumentos = dict(chamada.get("args") or {})
            else:
                nome = chamada.name
                argumentos = dict(chamada.args or {})
            self._log(f"AÇÃO {nome} {json.dumps(argumentos, ensure_ascii=False)}")
            payload = executar_ferramenta(nome, argumentos)
            self._log(f"OBSERVAÇÃO status={payload.get('status')}")
            acumuladas.append({
                "nome": nome,
                "args": argumentos,
                "status": payload.get("status", "error"),
            })
            parte = types.Part.from_function_response(
                name=nome,
                response={"output": payload},
            )
            identificador = getattr(chamada, "id", None)
            if identificador and parte.function_response is not None:
                parte.function_response.id = identificador
            partes.append(parte)
        return partes

    def _finalizar_modelo(
        self,
        contents: list[Any],
        chamadas: list[dict[str, Any]],
    ) -> dict[str, Any]:
        contents = [
            *contents,
            types.Content(
                role="user",
                parts=[types.Part.from_text(
                    text=(
                        "Produza o parecer final só com a pergunta e as observações "
                        "já recebidas. Não invente número. Se a evidência não basta, "
                        "use evidencia_insuficiente ou fora_da_amostra."
                    )
                )],
            ),
        ]
        config = config_de_geracao(
            system_instruction=PAPEL_DO_AGENTE,
            temperature=TEMPERATURA,
            response_mime_type="application/json",
            response_json_schema=DECISAO_SCHEMA,
        )
        resposta = _gerar_com_retry(contents, config)
        decisao = resposta.parsed
        if hasattr(decisao, "model_dump"):
            decisao = decisao.model_dump()
        if not isinstance(decisao, dict):
            decisao = json.loads(resposta.text)
        return decisao

    def executar(
        self,
        pergunta: str,
        decidir: Callable[[list[dict[str, Any]]], list[Any]] | None = None,
        gravar: bool = True,
    ) -> dict[str, Any]:
        """Percorre o ciclo. decidir=None usa o Gemini. Um callable substitui o modelo nos testes."""
        percepcao = self.perceber(pergunta)
        self._log("PERGUNTA " + percepcao["pergunta"])
        chamadas_feitas: list[dict[str, Any]] = []
        contents: list[Any] = [
            types.Content(
                role="user",
                parts=[types.Part.from_text(
                    text="PERGUNTA DO ANALISTA\n" + percepcao["pergunta"]
                )],
            )
        ]
        config_tools = None
        if decidir is None:
            config_tools = config_de_geracao(
                system_instruction=PAPEL_DO_AGENTE,
                temperature=TEMPERATURA,
                tools=[ferramenta_gemini()],
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                tool_config=types.ToolConfig(
                    function_calling_config=types.FunctionCallingConfig(mode="AUTO")
                ),
            )

        self.status = "DECIDING"
        for iteracao in range(1, self.max_iteracoes + 1):
            self._log(f"--- iteração {iteracao} ---")
            if decidir is None:
                resposta = _gerar_com_retry(contents, config_tools)
                if not resposta.candidates or resposta.candidates[0].content is None:
                    raise RuntimeError("O modelo não devolveu conteúdo.")
                contents.append(resposta.candidates[0].content)
                solicitadas = list(resposta.function_calls or [])
            else:
                solicitadas = decidir(chamadas_feitas)

            if not solicitadas:
                if decidir is None:
                    decisao = self._finalizar_modelo(contents, chamadas_feitas)
                else:
                    decisao = {
                        "resposta": "O ciclo encerrou sem nova ação.",
                        "grao": "indeterminado",
                        "fontes": [],
                        "conclusao": "evidencia_insuficiente",
                        "pendencias": ["Nenhuma ferramenta foi concluída com sucesso."],
                        "justificativa": "O modelo de teste parou antes de obter evidência.",
                    }
                problemas = validar_decisao(decisao, chamadas_feitas)
                return self._parecer(percepcao, decisao, chamadas_feitas, problemas, gravar)

            partes = self._executar_chamadas(solicitadas, chamadas_feitas)
            if decidir is None:
                # Os Flash recentes recusam o papel tool. O retorno da função volta como user.
                contents.append(types.Content(role="user", parts=partes))

        decisao = {
            "resposta": (
                f"O ciclo parou ao atingir {self.max_iteracoes} iterações. "
                "O parecer fica incompleto de propósito, para não continuar sem limite."
            ),
            "grao": "indeterminado",
            "fontes": sorted({chamada["nome"] for chamada in chamadas_feitas}),
            "conclusao": "limite_de_execucao",
            "pendencias": ["Aumentar o limite ou reformular a pergunta antes de concluir."],
            "justificativa": "O runtime interrompeu o laço. As chamadas já feitas permanecem no registro.",
        }
        problemas = validar_decisao(decisao, chamadas_feitas)
        return self._parecer(percepcao, decisao, chamadas_feitas, problemas, gravar)


print("Agente definido. MAX_ITERACOES =", MAX_ITERACOES)


---

# 8. Testes de comportamento do runtime

Estes testes não chamam o LLM. Eles verificam o que o enunciado pede do mecanismo: seleção e contrato das ferramentas, ausência, argumento inválido, falha capturada e respeito ao limite de iterações.


In [ ]:
def _exigir(condicao: bool, mensagem: str) -> None:
    if not condicao:
        raise AssertionError(mensagem)


def testar_conhecimento_encontra_iap() -> None:
    retorno = consultar_base_de_conhecimento("O que é IAP e por que lotação da perna não serve?")
    codigos = {item["codigo"] for item in retorno["evidencias"]}
    _exigir(retorno["status"] == "success", "conhecimento deveria devolver success")
    _exigir("IAP-001" in codigos, f"IAP-001 não veio na recuperação: {codigos}")


def testar_conhecimento_vazio() -> None:
    retorno = consultar_base_de_conhecimento("   ")
    _exigir(retorno["status"] == "invalid_arguments", "pergunta vazia deveria ser inválida")


def _vendas_caso() -> pd.DataFrame:
    return VENDAS[
        (VENDAS["linha"] == LINHA_AMOSTRA) & VENDAS["data_viagem"].str.startswith(DATA_AMOSTRA)
    ]


def _ocupacao_caso() -> pd.DataFrame:
    return OCUPACAO[
        (OCUPACAO["linha"] == LINHA_AMOSTRA) & OCUPACAO["data_viagem"].str.startswith(DATA_AMOSTRA)
    ]


def testar_vendas_do_recorte() -> None:
    retorno = consultar_vendas(LINHA_AMOSTRA, DATA_AMOSTRA)
    esperado = _vendas_caso()
    _exigir(retorno["status"] == "success", "vendas do caso deveriam existir")
    _exigir(retorno["quantidade_bilhetes"] == len(esperado), "quantidade divergente do CSV")
    _exigir(
        abs(retorno["receita_total"] - round(float(esperado["receita"].sum()), 2)) < 0.02,
        "receita divergente do CSV",
    )
    _exigir(retorno["grao"] == "bilhete", "grão da venda deveria ser bilhete")


def testar_vendas_fora_da_amostra() -> None:
    retorno = consultar_vendas("cidade-inexistente - destino-inexistente", DATA_AMOSTRA)
    _exigir(retorno["status"] == "not_found", "linha ausente deveria ser not_found")
    _exigir("receita_total" not in retorno, "not_found não pode trazer receita")


def testar_data_invalida() -> None:
    retorno = consultar_vendas(LINHA_AMOSTRA, "05/04/2026")
    _exigir(retorno["status"] == "invalid_arguments", "data fora do contrato deveria falhar")


def testar_ocupacao_do_recorte() -> None:
    retorno = consultar_ocupacao("belo horizonte - rio de janeiro", DATA_AMOSTRA)
    _exigir(retorno["status"] == "success", "ocupação do recorte deveria existir")
    _exigir(retorno["quantidade_pernas"] == len(_ocupacao_caso()), "contagem de pernas divergente")
    _exigir(retorno["iap"] == "nao_calculavel", "a ferramenta não pode inventar IAP")
    _exigir(retorno["pernas_mais_cheias"], "deveria apontar pernas para a pergunta de lotação")


def testar_ferramenta_desconhecida_e_falha() -> None:
    desconhecida = executar_ferramenta("consultar_erp", {"linha": LINHA_AMOSTRA})
    _exigir(desconhecida["status"] == "error", "nome fora do contrato deveria voltar como error")

    def instavel() -> dict[str, Any]:
        raise RuntimeError("falha transitória de leitura")

    FUNCOES["ferramenta_instavel"] = instavel
    SCHEMAS["ferramenta_instavel"] = {
        "type": "object",
        "additionalProperties": False,
        "properties": {},
    }
    try:
        captura = executar_ferramenta("ferramenta_instavel", {})
    finally:
        FUNCOES.pop("ferramenta_instavel", None)
        SCHEMAS.pop("ferramenta_instavel", None)
    _exigir(captura["status"] == "error", "exceção deveria virar observação de error")
    _exigir("falha transitória" in captura["observacao"], "a observação deveria preservar a causa")


def testar_limite_de_execucao() -> None:
    def sempre_vende(_historico: list[dict[str, Any]]) -> list[dict[str, Any]]:
        return [{"nome": "consultar_vendas", "args": {"linha": LINHA_AMOSTRA, "data_viagem": DATA_AMOSTRA}}]

    agente = AgenteAnalise(max_iteracoes=2, verbose=False)
    registro = agente.executar("Qual a receita?", decidir=sempre_vende, gravar=False)
    _exigir(registro["conclusao"] == "limite_de_execucao", "o laço deveria parar no limite")
    _exigir(len(registro["chamadas"]) == 2, "deveria executar só as iterações permitidas")
    _exigir(agente.status == "IDLE", "o agente deveria voltar a IDLE")


def testar_encerramento_sem_evidencia() -> None:
    agente = AgenteAnalise(max_iteracoes=3, verbose=False)
    registro = agente.executar("Pergunta sem ação", decidir=lambda _historico: [], gravar=False)
    _exigir(registro["conclusao"] == "evidencia_insuficiente", "parar sem ferramenta não responde o caso")
    _exigir(registro["problemas_de_validacao"] == [], "o parecer local deveria respeitar o contrato")


def testar_acao_grava_parecer() -> None:
    caminho = PASTA / "parecer_analista.csv"
    if caminho.exists():
        caminho.unlink()
    agente = AgenteAnalise(max_iteracoes=1, verbose=False)
    agente.executar("Pergunta de teste da ação", decidir=lambda _historico: [], gravar=True)
    _exigir(caminho.exists(), "agir() deveria criar parecer_analista.csv")
    _exigir("evidencia_insuficiente" in caminho.read_text(encoding="utf-8"), "o parecer deveria registrar a conclusão")
    caminho.unlink()


TESTES = [
    testar_conhecimento_encontra_iap,
    testar_conhecimento_vazio,
    testar_vendas_do_recorte,
    testar_vendas_fora_da_amostra,
    testar_data_invalida,
    testar_ocupacao_do_recorte,
    testar_ferramenta_desconhecida_e_falha,
    testar_limite_de_execucao,
    testar_encerramento_sem_evidencia,
    testar_acao_grava_parecer,
]
for teste in TESTES:
    teste()
    print("ok", teste.__name__)
print(f"{len(TESTES)} testes do runtime passaram.")


---

# 9. Cliente do modelo e índice vetorial

A partir daqui a célula precisa da chave. No Colab, crie o secret `GEMINI_API_KEY` e libere o acesso do notebook. Ela percorre os Flash da seção 5 até um responder. Se o embedding falhar, o agente continua pela recuperação lexical já testada.


In [ ]:
# REQUER_API
def obter_api_key() -> str:
    """Recupera a chave sem gravá-la no notebook."""
    try:
        from google.colab import userdata  # type: ignore

        chave = userdata.get("GEMINI_API_KEY")
        if chave:
            print("Chave obtida dos Secrets do Colab.")
            return chave
    except Exception:
        pass

    chave = os.environ.get("GEMINI_API_KEY", "").strip()
    if chave:
        print("Chave obtida da variável de ambiente GEMINI_API_KEY.")
        return chave

    from getpass import getpass

    chave = getpass("Cole sua GEMINI_API_KEY (a digitação fica oculta): ").strip()
    if not chave:
        raise RuntimeError("Nenhuma chave foi fornecida. Gere uma em https://aistudio.google.com/apikey")
    print("Chave obtida por digitação.")
    return chave


def _modelos_flash_candidatos() -> list[str]:
    """Junta a lista local com os Flash de texto que a conta enxerga."""
    candidatos = list(MODELOS_FLASH)
    try:
        remotos = []
        for modelo in CLIENTE.models.list():
            nome = str(getattr(modelo, "name", "") or "")
            nome = nome.removeprefix("models/")
            texto = nome.lower()
            if "flash" not in texto:
                continue
            if any(pedaço in texto for pedaço in (
                "image", "tts", "live", "audio", "embed", "transcribe", "native", "omni",
            )):
                continue
            remotos.append(nome)
        for nome in remotos:
            if nome not in candidatos:
                candidatos.append(nome)
    except Exception as erro:
        print(f"Não foi possível listar os modelos ({type(erro).__name__}). Seguindo a lista local.")
    return candidatos


def _erro_de_configuracao(erro: Exception) -> bool:
    texto = str(erro).lower()
    return any(pedaço in texto for pedaço in ("thinking", "budget", "not supported", "invalid argument", "unknown field"))


def escolher_modelo_flash() -> str:
    """Testa os Flash em ordem e fica com o primeiro que devolve texto."""
    global MODELO, USAR_ORCAMENTO_DE_RACIOCINIO
    falhas: list[str] = []
    for nome in _modelos_flash_candidatos():
        for com_orcamento in (True, False):
            USAR_ORCAMENTO_DE_RACIOCINIO = com_orcamento
            try:
                teste = CLIENTE.models.generate_content(
                    model=nome,
                    contents="Responda apenas com a palavra: pronto",
                    config=config_de_geracao(temperature=0.0, max_output_tokens=16),
                )
                texto = (teste.text or "").strip()
                if not texto:
                    raise RuntimeError("resposta vazia")
                MODELO = nome
                print(f"Modelo em uso: {MODELO}")
                print("Resposta de teste:", texto)
                if not com_orcamento:
                    print("Este modelo não aceitou thinking_budget. As próximas chamadas seguem sem ele.")
                return MODELO
            except Exception as erro:
                if com_orcamento and _erro_de_configuracao(erro):
                    continue
                falhas.append(f"{nome}: {type(erro).__name__}")
                print(f"Indisponível: {nome} ({type(erro).__name__})")
                break
    USAR_ORCAMENTO_DE_RACIOCINIO = True
    raise RuntimeError("Nenhum modelo Flash respondeu. " + " | ".join(falhas))


def preparar_cliente() -> str:
    """Cria o cliente, escolhe um Flash disponível e indexa o conhecimento."""
    global CLIENTE
    CLIENTE = genai.Client(api_key=obter_api_key())
    escolher_modelo_flash()
    return indexar_conhecimento()


MODO_CONHECIMENTO = preparar_cliente()
print("Modo de conhecimento ativo:", MODO_CONHECIMENTO)


---

# 10. Demonstração

A pergunta obriga o agente a separar regra e fato. Receita vem do bilhete. Lotação vem da perna. IAP fica como pendência, porque o ASK não está na amostra. O parecer é gravado em `parecer_analista.csv`.


In [ ]:
if CLIENTE is None:
    raise RuntimeError("Execute antes a célula que cria o cliente e indexa o conhecimento.")

PERGUNTA_DEMO = (
    "Na linha Belo Horizonte–Rio de Janeiro, em 2026-04-05, "
    "o que os números de receita e de lotação permitem dizer, "
    "e por que a lotação da perna não é o IAP?"
)

agente = AgenteAnalise(verbose=True)
parecer = agente.executar(PERGUNTA_DEMO)
print("\nPARECER")
print(json.dumps(
    {chave: valor for chave, valor in parecer.items() if chave != "chamadas"},
    ensure_ascii=False,
    indent=2,
))
print("Chamadas:", [(item["nome"], item["status"]) for item in parecer["chamadas"]])


---

# 11. Testes de comportamento com o modelo

Quatro perguntas curtas. O critério não é a redação: é a ferramenta escolhida, o tratamento de ausência e a conclusão.


In [ ]:
if CLIENTE is None:
    raise RuntimeError("Execute antes a célula que cria o cliente e indexa o conhecimento.")


def _usadas(registro: dict[str, Any]) -> set[str]:
    return {chamada["nome"] for chamada in registro["chamadas"]}


def testar_selecao_de_vendas() -> None:
    registro = AgenteAnalise(verbose=False).executar(
        "Qual a receita total de bilhetes da linha "
        "mg/belo horizonte - rj/rio de janeiro em 2026-04-05?",
        gravar=False,
    )
    _exigir("consultar_vendas" in _usadas(registro), f"não consultou vendas: {_usadas(registro)}")
    _exigir(registro["conclusao"] == "respondida", registro["conclusao"])
    _exigir(registro["problemas_de_validacao"] == [], str(registro["problemas_de_validacao"]))


def testar_selecao_de_ocupacao() -> None:
    registro = AgenteAnalise(verbose=False).executar(
        "Onde o ônibus encheu mais na linha Belo Horizonte–Rio de Janeiro em 2026-04-05?",
        gravar=False,
    )
    _exigir("consultar_ocupacao" in _usadas(registro), f"não consultou ocupação: {_usadas(registro)}")
    _exigir(registro["conclusao"] == "respondida", registro["conclusao"])


def testar_selecao_de_regra() -> None:
    registro = AgenteAnalise(verbose=False).executar(
        "O que é IAP e por que não posso obter a receita somando as pernas?",
        gravar=False,
    )
    _exigir(
        "consultar_base_de_conhecimento" in _usadas(registro),
        f"não consultou conhecimento: {_usadas(registro)}",
    )
    _exigir(registro["conclusao"] == "respondida", registro["conclusao"])


def testar_ausencia() -> None:
    registro = AgenteAnalise(verbose=False).executar(
        "Qual a receita da linha Manaus–Porto Velho em 2026-04-05?",
        gravar=False,
    )
    _exigir("consultar_vendas" in _usadas(registro), "deveria consultar antes de declarar ausência")
    _exigir(registro["conclusao"] in {"fora_da_amostra", "evidencia_insuficiente"}, registro["conclusao"])
    _exigir(registro["conclusao"] != "respondida", "ausência não pode ser conclusão respondida")


TESTES_MODELO = [
    testar_selecao_de_vendas,
    testar_selecao_de_ocupacao,
    testar_selecao_de_regra,
    testar_ausencia,
]
for teste in TESTES_MODELO:
    teste()
    print("ok", teste.__name__)
print(f"{len(TESTES_MODELO)} testes com o modelo passaram.")
